# Canonical Colab Pro notebook for the ResNet-18 comparison

**Before Run all, complete these three steps**

1. **Runtime → Change runtime type → select a GPU** (T4 is enough; this notebook refuses to train without CUDA and has no CPU fallback).
2. In the flag cell below, set **`run_training = True`**.
3. **Run all**, and approve the Google Drive mount prompt when it appears.

Six runs (baseline/robust × seeds 42/43/44) take roughly 2 to 3 hours total and run unattended. Every completed run is validated and published atomically to `MyDrive/ECS7037P/resnet18_results/` on Drive, so a dropped session keeps its finished bundles and a rerun trains only what is missing. When all six bundles validate, `resnet18_all_results.csv` is written beside them; download the whole Drive folder back into the repo's `results/` directory afterwards.

This is the second architecture for the CIFAKE robustness comparison. It answers both halves of the marker's request. The proposal feedback explicitly asks for a wider range of architectures and transfer learning. The model is ResNet-18 pretrained on ImageNet with the stem adapted for 32x32 input; He et al. (2016), section 4.2, use exactly this 3x3 no-max-pool stem for their CIFAR-10 networks. Everything the CNN runs froze stays frozen here. This includes the 90/10 seeded split, seeds 42/43/44 paired per strategy, Adam with up to 30 epochs and patience 5 against a clean validation set, the CIFAR-10-C severity constants from Hendrycks and Dietterich (2019), blur held out of training, the 16-condition five-metric evaluation, and the same train() function.

I make one deliberate departure for the methodology section. The learning rate is 1e-4 rather than the CNN's 1e-3 because fine-tuning pretrained weights at 1e-3 can disturb features that are already good. It is identical across both strategies, so the within-architecture comparison is unaffected. Normalisation uses CIFAR-10 channel statistics for every model including this pretrained one. These statistics are slightly mismatched to the ImageNet statistics used to train the backbone, but consistency across architectures matters more for a fair comparison. That mismatch and the discarded pretrained conv1 weights are limitations for the report, not reasons to change the protocol.

With `run_training = False` (the default), the notebook is a verification session. Setup and the structural checks run everywhere without a dataset, network, or GPU. They prove the CIFAR stem, the two-class head, a 32x32 forward pass, the publication pipeline, and that no ResNet artefact path can collide with a CNN bundle. All ResNet artefacts carry the `resnet18_` prefix.


In [1]:
import os
import sys
import copy
import json
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms
from torchvision.io import encode_jpeg, decode_jpeg
import torchvision.transforms.functional as TF
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score)
from torch.utils.data import DataLoader, random_split

In [2]:
#Colab environment gate. This is the canonical Colab Pro runner: training
#requires CUDA and persists to Google Drive. Run anywhere else it degrades to
#a verification session - structural checks only, training refused.
try:
  import google.colab
  IN_COLAB = True
except ImportError:
  IN_COLAB = False

#The only dependency missing from a stock Colab image is kagglehub; pin it to
#the version the local verification used. Nothing else is (re)installed:
#torch and torchvision ship CUDA-built on the image, and reinstalling them
#risks a broken driver pairing.
KAGGLEHUB_PIN = "1.0.2"
try:
  import kagglehub
except ImportError:
  import subprocess
  subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", f"kagglehub=={KAGGLEHUB_PIN}"],
    check=True)
  import kagglehub
print(f"kagglehub {kagglehub.__version__}")

if IN_COLAB:
  #Fail loudly: a Colab session without a GPU runtime is a misconfiguration,
  #not a slower option.
  if not torch.cuda.is_available():
    raise RuntimeError(
      "No CUDA GPU visible. Runtime > Change runtime type > select a GPU, "
      "then Run all again. This runner never falls back to CPU.")
  print(f"gpu: {torch.cuda.get_device_name(0)}, cuda {torch.version.cuda}, "
      f"cudnn {torch.backends.cudnn.version()}, torch {torch.__version__}, "
      f"torchvision {torchvision.__version__}")

  from google.colab import drive
  drive.mount("/content/drive")
  #Dedicated results directory on Drive: bundles survive the session, and the
  #write-once gate makes an interrupted matrix resumable.
  DRIVE_RESULTS_DIR = "/content/drive/MyDrive/ECS7037P/resnet18_results"
  os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)
  print(f"results persist to {DRIVE_RESULTS_DIR}")
else:
  DRIVE_RESULTS_DIR = None
  print("not Colab: verification session, training is refused")


kagglehub 1.0.2
not Colab: verification session, training is refused


In [3]:
#Run training set to false here by default: the notebook is then a pure
#verification session. Set to true (on Colab, with a GPU runtime) to run the
#full matrix; the structural checks at the bottom run either way and the
#dataset is only downloaded when training actually starts.
run_training = False

#Scope: the full matrix, seeds 42/43/44, both strategies paired within each
#seed. Bundles are write-once and the resume gate skips any pair that already
#validates, so a rerun after an interruption only trains what is missing; the
#aggregate is only ever written once all six bundles validate.


In [4]:
LABEL_REAL = 0
LABEL_FAKE = 1


#Corruption levels. From Hendryks and Dietterich (2019),
#CIFAR-10-C: https://github.com/hendrycks/robustness/blob/master/ImageNet-C/create_c/make_cifar_c.py
#Identical to the CNN notebook; frozen, do not touch.

SEVERITIES = {
  "noise": [0.04, 0.06, 0.08, 0.09, 0.10],   #Gaussian sigma
  "blur":  [0.4, 0.6, 0.7, 0.8, 1.0],        #Gaussian sigma
  "jpeg":  [80, 65, 58, 50, 40],             #JPEG quality
}

AUGMENT_KINDS = ["noise", "jpeg"]
HELDOUT_KINDS = ["blur"]

CONDITIONS = [("none", 0.0)] + [
  (kind, sev) for kind, levels in SEVERITIES.items() for sev in levels
]

#Standard statistics for CIFAR-10 RGB channels which CIFAKE is made from.
NORM_MEAN = (0.4914, 0.4822, 0.4465)
NORM_STD = (0.2470, 0.2435, 0.2616)


def get_device():
  """CUDA, and in Colab nothing else: this runner fails loudly rather than
  falling back to CPU. Off Colab a verification session may sit on mps/cpu,
  but the run gate refuses to train there."""
  if torch.cuda.is_available():
    return torch.device("cuda")
  if IN_COLAB:
    raise RuntimeError("CUDA unavailable in Colab: select a GPU runtime")
  if torch.backends.mps.is_available():
    return torch.device("mps")
  return torch.device("cpu")


DEVICE = get_device()
print(f"device: {DEVICE}")


is_seed_set = False  #train() checks this so an unseeded run shows a warning


def set_seed(seed, deterministic=True):
  """Seed every random number generator. Call at start of each experiment."""
  global is_seed_set
  is_seed_set = True
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  torch.cuda.manual_seed_all(seed)
  torch.backends.cudnn.deterministic = deterministic
  torch.backends.cudnn.benchmark = not deterministic


def flip_label(y):
  #Module level, not a closure: spawn-start DataLoader workers cannot pickle
  #a nested function. CIFAKE folders sort FAKE first, so flip to FAKE=1.
  return 1 - y

device: mps


In [5]:
CIFAKE_SLUG = "cifake-real-and-ai-generated-synthetic-images"
KAGGLE_MOUNT = f"/kaggle/input/{CIFAKE_SLUG}"
CIFAKE_CLASS_DIRS = (("train", "REAL"), ("train", "FAKE"),
           ("test", "REAL"), ("test", "FAKE"))


def verify_cifake_layout(root):
  """The four class directories CIFAKE must expose; a mount or download that
  lacks any of them is malformed and fails before a single image is read."""
  missing = [os.path.join(split, cls) for split, cls in CIFAKE_CLASS_DIRS
       if not os.path.isdir(os.path.join(root, split, cls))]
  if missing:
    raise RuntimeError(
      f"CIFAKE layout invalid at {root}: missing {', '.join(missing)}")


def dataset_meta_for_root(root):
  """Classify a CIFAKE root and pin its provenance.

  Two legitimate forms exist. KaggleHub caches end in versions/N and carry the
  release number, which we require to be 3 (the release the CNN bundles used).
  Colab's Kaggle mount at /kaggle/input/<slug> exposes the same files but no
  version directory, so version identity is genuinely unavailable there: it is
  recorded as "unversioned" rather than assumed to be 3. Validation compares
  the recorded value strictly, so unversioned-mount bundles and versioned
  kagglehub bundles never silently mix.
  """
  verify_cifake_layout(root)

  parent = os.path.basename(os.path.dirname(root))
  if parent == "versions":
    version = os.path.basename(root)
    assert version == "3", f"expected CIFAKE release 3, got versions/{version}"
    return {"cifake_version": version, "provenance": "kagglehub_versioned",
        "version_available": True, "path": root}

  if os.path.basename(root) == CIFAKE_SLUG:
    return {"cifake_version": "unversioned", "provenance": "kaggle_colab_mount",
        "version_available": False, "path": root}

  raise RuntimeError(
    f"unrecognised CIFAKE root {root}: neither a kagglehub versions/N cache "
    f"nor a /kaggle/input/{CIFAKE_SLUG} mount")


def find_data_root():
  """Locate CIFAKE: Colab's Kaggle mount when present, else the kagglehub
  cache (downloading only if absent).

  The kagglehub import sits inside this function so that structural checks and
  fresh-kernel runs with run_training = False never touch the network. Returns
  (root, meta) where meta pins path, provenance and version availability.
  """
  if os.path.isdir(KAGGLE_MOUNT):
    root = KAGGLE_MOUNT
  else:
    import kagglehub
    root = kagglehub.dataset_download(f"birdy654/{CIFAKE_SLUG}")
  meta = dataset_meta_for_root(root)
  print(f"CIFAKE at {meta['path']} ({meta['provenance']}, "
      f"version {meta['cifake_version']})")
  return root, meta


def get_dataloaders(data_root, seed, batch_size=128, val_frac=0.1, num_workers=None):
  """Loads CIFAKE and returns the train, validation and test loaders.

  The same seed always gives the same train/val split as the CNN runs, because
  random_split is driven by a generator seeded with the same value; the split
  does not depend on the model. Images come back unnormalised in [0, 1]
  because corruption has to be applied first.
  """
  if num_workers is None:
    #Disk I/O bottlenecks GPU training without workers (178s vs 28s per epoch
    #on the CNN); cap at 4 so a 2-vCPU Colab box is not oversubscribed badly.
    num_workers = min(4, os.cpu_count() or 1) if sys.platform.startswith("linux") else 0

  to_tensor = transforms.ToTensor()

  full_train = torchvision.datasets.ImageFolder(
    os.path.join(data_root, "train"), transform=to_tensor, target_transform=flip_label)
  test_set = torchvision.datasets.ImageFolder(
    os.path.join(data_root, "test"), transform=to_tensor, target_transform=flip_label)

  n_val = int(len(full_train) * val_frac)
  n_train = len(full_train) - n_val
  train_set, val_set = random_split(
    full_train, [n_train, n_val],
    generator=torch.Generator().manual_seed(seed),
  )

  #pin_memory is a CUDA-only optimisation; MPS warns and ignores it.
  pin = DEVICE.type == "cuda"
  train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=pin)
  val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin)
  test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin)
  return train_loader, val_loader, test_loader


def normalise(images):
  """Channel normalisation, always after corruption."""
  return TF.normalize(images, NORM_MEAN, NORM_STD)


def corrupt(images, kind, severity):
  """Apply one degradation to a batch; identical operators to the CNN notebook."""
  if kind == "none" or severity is None:
    return images

  if kind == "blur":
    k = int(2 * np.ceil(3 * severity) + 1)
    return TF.gaussian_blur(images, kernel_size=[k, k], sigma=[severity, severity])

  if kind == "noise":
    noisy = images + torch.randn_like(images) * severity
    return noisy.clamp(0.0, 1.0)

  if kind == "jpeg":
    #encode_jpeg needs uint8 CHW tensors on CPU, one image at a time.
    device = images.device
    as_uint8 = (images.clamp(0, 1) * 255).to(torch.uint8).cpu()
    out = torch.stack([
      decode_jpeg(encode_jpeg(img, quality=int(severity)))
      for img in as_uint8
    ])
    return (out.float() / 255.0).to(device)

  raise ValueError(f"Unknown corruption kind: {kind}")


def augment_batch(images, kinds=None, rng=None, n_chunks=8):
  """Training-time corruption mix: 50% h-flip, then chunks of the batch each
  get a random kind at a random severity. Identical to the CNN notebook."""
  if rng is None:
    rng = random
  if kinds is None:
    kinds = AUGMENT_KINDS

  n = images.size(0)
  out = images.clone()

  flip = torch.rand(n, device=images.device) < 0.5
  out[flip] = torch.flip(out[flip], dims=[3])

  perm = torch.randperm(n)
  for chunk in torch.chunk(perm, min(n_chunks, n)):
    kind = rng.choice(kinds)
    severity = rng.choice(SEVERITIES[kind])
    out[chunk] = corrupt(out[chunk], kind, severity)
  return out

In [6]:
def build_resnet18_cifar(pretrained=True):
  """ResNet-18 pretrained on ImageNet, with the stem adapted for 32x32 input.

  weights='DEFAULT' because the proposal feedback names transfer learning
  explicitly: the question becomes whether ImageNet features transfer to
  AI-artifact detection, and whether pretraining buys robustness or only
  clean accuracy. The default 7x7 stride-2 stem plus max-pool is built for
  224x224 inputs and would collapse a 32x32 image 4x before the first
  residual block, so: 3x3 stride-1 conv, no max-pool - the stem He et al.
  (2016, section 4.2) use for their CIFAR-10 networks. The swap discards the
  pretrained conv1 weights, so later layers see features at a slightly
  different scale than they were trained on (a limitation for the report);
  everything after the stem keeps its ImageNet weights. Two-class head,
  freshly initialised.

  pretrained=False exists for the structural checks and the checkpoint
  shape-gate, which need the architecture but not the weights and must not
  touch the network.
  """
  model = torchvision.models.resnet18(weights="DEFAULT" if pretrained else None)

  model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
  model.maxpool = nn.Identity()
  model.fc = nn.Linear(512, 2)

  #Frozen stem, asserted so a drive-by edit cannot silently change the protocol.
  assert isinstance(model.conv1, nn.Conv2d)
  assert model.conv1.kernel_size == (3, 3)
  assert model.conv1.stride == (1, 1)
  assert model.conv1.padding == (1, 1)
  assert model.conv1.bias is None
  assert isinstance(model.maxpool, nn.Identity)
  assert model.fc.out_features == 2 and model.fc.in_features == 512
  return model


In [7]:
DEFAULT_CONFIG = {
  "lr": 1e-4,  #fine-tuning rate; 1e-3 can disturb pretrained weights
  "weight_decay": 0.0,
  "max_epochs": 30,
  "patience": 5,
  "augment": False,
  "optimiser": "adam",
  "augment_kinds": list(AUGMENT_KINDS),
  "val_mode": "clean",
}


def train(model, train_loader, val_loader, config=None, verbose=True):
  """Trains a model with early stopping; the taught loop, under the same
  shared recipe as the CNN runs, apart from the fine-tuning learning rate. Gaps measured under this recipe can reflect
  how well the recipe suits each architecture, not architecture alone."""
  cfg = DEFAULT_CONFIG.copy()
  if config is not None:
    cfg.update(config)

  if not is_seed_set:
    print("WARNING: set_seed() has not been called. This run is not reproducible.")

  model = model.to(DEVICE)
  criterion = nn.CrossEntropyLoss()

  if cfg["optimiser"] == "adam":
    optimiser = torch.optim.Adam(
      model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
  else:
    raise ValueError(f"Unknown optimiser: {cfg['optimiser']}")

  best_loss = float("inf")
  best_state = copy.deepcopy(model.state_dict())
  best_epoch = 0
  epochs_since_improvement = 0
  train_losses = []
  val_losses = []

  for epoch in range(1, cfg["max_epochs"] + 1):
    model.train()
    running = 0.0

    for images, labels in train_loader:
      images, labels = images.to(DEVICE), labels.to(DEVICE)

      if cfg["augment"]:
        images = augment_batch(images, cfg["augment_kinds"])

      images = normalise(images)  #always after corruption

      optimiser.zero_grad()
      outputs = model(images)
      loss = criterion(outputs, labels)
      loss.backward()
      optimiser.step()
      running += loss.item() * images.size(0)

    train_loss = running / len(train_loader.dataset)

    #val_mode is clean for every run in this matrix: both strategies early-stop
    #against the same yardstick, so epoch budgets stay comparable.
    model.eval()
    running = 0.0
    with torch.no_grad():
      for images, labels in val_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        images = normalise(images)
        outputs = model(images)
        loss = criterion(outputs, labels)
        running += loss.item() * images.size(0)
    val_loss = running / len(val_loader.dataset)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    if verbose:
      print(f"epoch {epoch:2d}  train {train_loss:.4f}  val {val_loss:.4f}")

    if val_loss < best_loss:
      best_loss = val_loss
      best_epoch = epoch
      best_state = copy.deepcopy(model.state_dict())
      epochs_since_improvement = 0
    else:
      epochs_since_improvement += 1
      if epochs_since_improvement >= cfg["patience"]:
        if verbose:
          print(f"early stop at epoch {epoch} (best was {best_epoch})")
        break

  model.load_state_dict(best_state)  #always return the best, not the last

  history = {
    "epochs_run": len(train_losses),
    "best_epoch": best_epoch,
    "best_val_loss": best_loss,
    "train_losses": train_losses,
    "val_losses": val_losses,
    "config": cfg,
  }
  return model, history


def evaluate(model, loader, kind="none", severity=0.0):
  """Tests a model on one corruption condition; five metrics, fakes positive."""
  model = model.to(DEVICE).eval()
  all_probs, all_preds, all_labels = [], [], []

  with torch.no_grad():
    for images, labels in loader:
      images = images.to(DEVICE)
      images = corrupt(images, kind, severity)  #corrupt before normalising
      images = normalise(images)
      outputs = model(images)
      probs = torch.softmax(outputs, dim=1)[:, LABEL_FAKE]
      all_probs.append(probs.cpu())
      all_preds.append(outputs.argmax(dim=1).cpu())
      all_labels.append(labels)

  y_prob = torch.cat(all_probs).numpy()
  y_pred = torch.cat(all_preds).numpy()
  y_true = torch.cat(all_labels).numpy()

  return {
    "accuracy": accuracy_score(y_true, y_pred),
    "precision": precision_score(y_true, y_pred, pos_label=LABEL_FAKE, zero_division=0),
    "recall": recall_score(y_true, y_pred, pos_label=LABEL_FAKE, zero_division=0),
    "f1": f1_score(y_true, y_pred, pos_label=LABEL_FAKE, zero_division=0),
    "roc_auc": roc_auc_score(y_true, y_prob),
  }


def evaluate_all_conditions(model, test_loader, model_name, strategy, seed, history):
  """Every corruption condition for one trained model, as one dataframe.

  The trailing columns (epochs_run, best_epoch, lr, augment) mirror the CNN
  CSVs exactly, so the two architectures concatenate into one frame for the
  combined analysis without schema holes."""
  cfg = history["config"]
  rows = []
  for kind, severity in CONDITIONS:
    metrics = evaluate(model, test_loader, kind, severity)
    rows.append({
      "model": model_name, "strategy": strategy, "corruption": kind,
      "severity": severity, "seed": seed, **metrics,
      "epochs_run": history["epochs_run"], "best_epoch": history["best_epoch"],
      "lr": cfg["lr"], "augment": cfg["augment"],
    })
  return pd.DataFrame(rows)

In [8]:
#Experiment matrix. Identical to the CNN protocol except the architecture.
SEEDS = [42, 43, 44]

EXPERIMENT_CONFIG = {
  "lr": 1e-4,  #fine-tuning rate, identical across strategies; the one departure from the CNN recipe
  "weight_decay": 0.0,
  "max_epochs": 30,
  "patience": 5,
  "optimiser": "adam",
  "augment_kinds": list(AUGMENT_KINDS),
  "val_mode": "clean",
}

STRATEGIES = {
  "baseline": {"augment": False},
  "robust": {"augment": True},
}

assert all(set(o) == {"augment"} for o in STRATEGIES.values())
assert AUGMENT_KINDS == ["noise", "jpeg"] and HELDOUT_KINDS == ["blur"]
assert set(AUGMENT_KINDS).isdisjoint(HELDOUT_KINDS)

#On Colab every artefact lands in the dedicated Drive directory so it
#survives the session; locally the repo results/ is used for the collision
#checks against the existing CNN bundles.
RESULTS_DIR = DRIVE_RESULTS_DIR if IN_COLAB else "results"
MODEL_NAME = "ResNet18"
ARTEFACT_PREFIX = "resnet18_"
METRIC_COLS = ["accuracy", "precision", "recall", "f1", "roc_auc"]
VAL_FRAC = 0.1  #90/10 train/val split, frozen across every run


def run_paths(strategy, seed):
  """Deterministic artefact names, always under the resnet18_ prefix so a
  ResNet write can never land on a CNN bundle."""
  stem = f"{ARTEFACT_PREFIX}{strategy}_seed{seed}"
  return (os.path.join(RESULTS_DIR, f"{stem}.pt"),
      os.path.join(RESULTS_DIR, f"{stem}_history.json"),
      os.path.join(RESULTS_DIR, f"{stem}_results.csv"))


def assert_safe_path(path):
  """Overwrite protection by construction: every artefact this notebook
  writes must carry the resnet18_ prefix. A CNN filename fails loudly."""
  assert os.path.basename(path).startswith(ARTEFACT_PREFIX), (
    f"refusing to write {path}: not a {ARTEFACT_PREFIX} artefact")


def assert_write_once(path):
  """Run artefacts are write-once: an existing file is never overwritten,
  whatever state it is in. Delete manually to rerun."""
  assert_safe_path(path)
  assert not os.path.exists(path), f"refusing to overwrite existing artefact {path}"


def environment_meta(model):
  """Record the Colab hardware and library versions beside each run; the CNN
  bundles came from Apple-silicon MPS, so the report needs this to state the
  hardware difference honestly."""
  return {
    "device": str(DEVICE),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "python": sys.version.split()[0],
    "cudnn_deterministic": torch.backends.cudnn.deterministic,
    "param_count": sum(p.numel() for p in model.parameters()),
    "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
  }


def validate_bundle(ckpt_path, hist_path, csv_path, strategy, seed,
          expected_config, dataset_meta):
  """The full bundle gate against explicit paths, so publication can validate
  temporary files before renaming them to final names."""
  missing = [p for p in (ckpt_path, hist_path, csv_path) if not os.path.exists(p)]
  if missing:
    return None, f"missing {', '.join(os.path.basename(p) for p in missing)}"

  try:
    with open(hist_path) as f:
      history = json.load(f)
  except (json.JSONDecodeError, OSError) as err:
    return None, f"unreadable history ({err})"

  saved = history.get("config", {})
  drift = {k: (saved.get(k), v) for k, v in expected_config.items()
       if saved.get(k) != v}
  if drift:
    return None, f"config drift {drift}"

  saved_meta = history.get("dataset", {})
  meta_drift = {k: (saved_meta.get(k), v) for k, v in dataset_meta.items()
        if saved_meta.get(k) != v}
  if meta_drift:
    return None, f"dataset drift {meta_drift}"

  df = pd.read_csv(csv_path)
  if df.shape[0] != 16:
    return None, f"csv has {df.shape[0]} rows, expected 16"
  saved_conditions = set(zip(df["corruption"], df["severity"]))
  if saved_conditions != set(CONDITIONS):
    return None, "csv conditions do not match the 16-condition protocol"
  if not ((df["strategy"] == strategy).all() and (df["seed"] == seed).all()
      and (df["model"] == MODEL_NAME).all()):
    return None, "csv strategy/seed/model does not match its filename"
  if df[METRIC_COLS].isna().any().any():
    return None, "csv has missing metric values"
  if not ((df[METRIC_COLS] >= 0) & (df[METRIC_COLS] <= 1)).all().all():
    return None, "csv metric out of range"

  #Weights must load into the frozen stem with strict=True, which proves the
  #checkpoint was produced by this exact architecture. pretrained=False: the
  #gate needs the architecture only, and must stay network-free.
  try:
    state = torch.load(ckpt_path, map_location="cpu", weights_only=True)
    build_resnet18_cifar(pretrained=False).load_state_dict(state, strict=True)
  except Exception as err:
    return None, f"checkpoint does not load ({err})"

  return df, None


def check_saved_run(strategy, seed, expected_config, dataset_meta):
  """Accept a saved ResNet run only if the full bundle at its final paths is
  present and consistent; the CNN notebook's gate, parametrised for this model."""
  ckpt_path, hist_path, csv_path = run_paths(strategy, seed)
  return validate_bundle(ckpt_path, hist_path, csv_path, strategy, seed,
             expected_config, dataset_meta)


def preflight_publication(strategy, seed):
  """Before any compute is spent: every final and temporary artefact path for
  this run must be safe to create. Fails closed on leftovers."""
  finals = run_paths(strategy, seed)
  temps = tuple(p + ".tmp" for p in finals)
  for p in finals + temps:
    assert_write_once(p)
  return finals, temps


def quarantine_temps(temps, strategy, seed, reason):
  """Preserve failed-publication artefacts for inspection instead of deleting
  a checkpoint that cost real compute. Files land in a timestamped directory
  under resnet18_quarantine/ with the failure reason beside them. Quarantined
  files are never resumable and never valid: resume and the bundle gate look
  only at the final run_paths names, which quarantine can never contain."""
  remaining = [t for t in temps if os.path.exists(t)]
  if not remaining:
    return None
  stamp = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
  qdir = os.path.join(RESULTS_DIR, "resnet18_quarantine",
            f"{stamp}_{strategy}_seed{seed}")
  os.makedirs(qdir, exist_ok=True)
  for tmp in remaining:
    os.replace(tmp, os.path.join(qdir, os.path.basename(tmp)))
  with open(os.path.join(qdir, "REASON.txt"), "w") as f:
    f.write(reason + "\n")
  return qdir


def publish_bundle(strategy, seed, state_dict, history, df,
          expected_config, dataset_meta):
  """Crash-safe publication: write the whole bundle to temporary files,
  validate it there, then rename each file to its final name.

  Failure semantics, precisely: a validation or publication failure raised
  in-process moves every remaining temporary file to a quarantine directory
  with its reason, so a completed checkpoint is preserved for inspection, not
  deleted. A hard crash (kernel death) loses only the in-progress run, whose
  outputs were still in memory; any orphaned .tmp files it leaves are refused
  by the next session's preflight. Each os.replace is atomic, but the three
  renames together are not: a crash between renames leaves a partial bundle
  at the final names, which the fail-closed resume refuses. No final path is
  ever written directly, so a truncated file can never appear under a final
  name."""
  (ckpt_path, hist_path, csv_path), temps = preflight_publication(strategy, seed)
  ckpt_tmp, hist_tmp, csv_tmp = temps

  try:
    torch.save(state_dict, ckpt_tmp)
    with open(hist_tmp, "w") as f:
      json.dump(history, f)
    df.to_csv(csv_tmp, index=False)

    checked_df, reason = validate_bundle(ckpt_tmp, hist_tmp, csv_tmp, strategy,
                       seed, expected_config, dataset_meta)
    if checked_df is None:
      raise RuntimeError(f"bundle failed validation ({reason})")

    for tmp, final in zip(temps, (ckpt_path, hist_path, csv_path)):
      assert not os.path.exists(final), f"{final} appeared during the run"
      os.replace(tmp, final)
  except BaseException as err:
    qdir = quarantine_temps(temps, strategy, seed, f"{strategy} seed {seed}: {err}")
    if qdir is not None:
      raise RuntimeError(
        f"{strategy} seed {seed}: publication failed ({err}); remaining "
        f"temporary artefacts preserved in {qdir} for inspection only, "
        "quarantined files are never resumed or treated as valid.") from err
    raise


def write_aggregate(results):
  """Atomically replace the derived aggregate. A stale .tmp from a crashed
  earlier session is discarded explicitly: the aggregate is derived data,
  rebuilt in full from validated bundles, so a leftover temp holds nothing
  worth quarantining."""
  combined = os.path.join(RESULTS_DIR, f"{ARTEFACT_PREFIX}all_results.csv")
  combined_tmp = combined + ".tmp"
  assert_safe_path(combined)
  assert_safe_path(combined_tmp)
  if os.path.exists(combined_tmp):
    print(f"discarding stale aggregate temp {os.path.basename(combined_tmp)}")
    os.remove(combined_tmp)
  results.to_csv(combined_tmp, index=False)
  os.replace(combined_tmp, combined)
  return combined


def resolve_existing_bundle(strategy, seed, config, dataset_meta):
  """Fail-closed resume: a valid bundle is reused, an absent one trains, and
  a partial or invalid one raises so nothing is ever overwritten."""
  saved_df, reason = check_saved_run(strategy, seed, config, dataset_meta)
  if saved_df is not None:
    return saved_df
  present = [p for p in run_paths(strategy, seed) if os.path.exists(p)]
  if present:
    raise RuntimeError(
      f"{strategy} seed {seed}: existing artefacts failed validation ({reason}). "
      f"Refusing to overwrite {[os.path.basename(p) for p in present]}; "
      "inspect and delete them manually to rerun this pair.")
  return None


def run_experiments(seeds=None):
  """Train and evaluate every strategy x seed pair, publishing each bundle
  atomically once it validates. Baseline runs before robust within each seed.
  A hard crash loses only the in-progress run's compute (its outputs are in
  memory until publication); an in-process failure quarantines the finished
  temporaries instead. The fail-closed resume refuses any partial bundle a
  crash between renames might leave."""
  data_root, root_meta = find_data_root()
  #Gated keys: version identity and split fraction. Path and provenance are
  #recorded beside them for the report but not gated, because the same bundle
  #must validate on Colab and locally, where paths necessarily differ.
  dataset_meta = {"cifake_version": root_meta["cifake_version"], "val_frac": VAL_FRAC}
  dataset_record = {**dataset_meta,
            "provenance": root_meta["provenance"],
            "version_available": root_meta["version_available"],
            "path": root_meta["path"]}

  seeds = SEEDS if seeds is None else seeds
  frames = []

  for seed in seeds:
    for strategy, overrides in STRATEGIES.items():
      config = {**EXPERIMENT_CONFIG, **overrides}
      saved_df = resolve_existing_bundle(strategy, seed, config, dataset_meta)

      if saved_df is not None:
        print(f"{strategy} seed {seed}: complete saved run found, skipping")
        frames.append(saved_df)
        continue

      print(f"\n{MODEL_NAME} {strategy} seed {seed}")
      os.makedirs(RESULTS_DIR, exist_ok=True)
      preflight_publication(strategy, seed)  #fail before compute, not after

      set_seed(seed)
      train_loader, val_loader, test_loader = get_dataloaders(
        data_root, seed=seed, val_frac=VAL_FRAC)
      #Built after set_seed, so baseline and robust start from identical
      #weights within this architecture. Not comparable to the CNN's init.
      model = build_resnet18_cifar()

      t0 = time.time()
      model, history = train(model, train_loader, val_loader, config)
      history["wall_time_s"] = round(time.time() - t0, 1)
      history["dataset"] = dict(dataset_record)
      history["environment"] = environment_meta(model)

      #Training and evaluation outputs stay in memory until the whole bundle
      #is complete; nothing lands under a final name unvalidated.
      df = evaluate_all_conditions(model, test_loader, MODEL_NAME, strategy, seed, history)
      publish_bundle(strategy, seed, model.state_dict(), history, df,
             config, dataset_meta)
      frames.append(df)

  #The aggregate is rebuilt only from all six validated bundles, re-read from
  #disk. A staged subset session leaves the aggregate untouched rather than
  #replacing it with fewer rows.
  full_frames, incomplete = [], []
  for seed in SEEDS:
    for strategy, overrides in STRATEGIES.items():
      config = {**EXPERIMENT_CONFIG, **overrides}
      df, reason = check_saved_run(strategy, seed, config, dataset_meta)
      if df is None:
        incomplete.append(f"{strategy} seed {seed} ({reason})")
      else:
        full_frames.append(df)

  if incomplete:
    print(f"aggregate not written: {len(full_frames)}/6 bundles valid; "
        f"missing {'; '.join(incomplete)}")
    return pd.concat(frames, ignore_index=True) if frames else None

  results = pd.concat(full_frames, ignore_index=True)
  assert results.shape[0] == len(SEEDS) * len(STRATEGIES) * 16
  write_aggregate(results)
  print(f"aggregate written from all six validated bundles: {results.shape[0]} rows")
  return results


#This session's staged scope, stated once and read by the summary cell too.
STAGED_SEEDS = [42, 43, 44]

if run_training:
  #No CPU fallback, stated twice on purpose: the device cell requires CUDA in
  #Colab, and this gate refuses to spend hours training anywhere else.
  assert DEVICE.type == "cuda", f"training requires CUDA, got {DEVICE}"
  all_results = run_experiments(seeds=STAGED_SEEDS)  #the full matrix
else:
  print(f"run_training = False, experiments not started (staged scope: seeds {STAGED_SEEDS})")

run_training = False, experiments not started (staged scope: seeds [42, 43, 44])


In [9]:
#Structural checks. These run on every execution, dataset-free, and prove the
#frozen stem, the forward pass, paired initialisation, the corruption
#operators, and that no ResNet artefact path can touch a CNN bundle.

#1. Stem and head are exactly the frozen protocol (asserts also live in the builder).
set_seed(42)
model = build_resnet18_cifar(pretrained=False)  #architecture facts need no weight download
assert model.conv1.kernel_size == (3, 3) and model.conv1.stride == (1, 1)
assert model.conv1.bias is None
assert isinstance(model.maxpool, nn.Identity)
assert model.fc.out_features == 2
n_params = sum(p.numel() for p in model.parameters())
print(f"stem ok: conv1 3x3 stride 1 no bias, no max-pool, 2-class head, {n_params:,} parameters")

#2. Untrained forward pass at native 32x32, on CPU so the check is
#device-independent. eval + no_grad, otherwise train-mode batch norm would
#update its running statistics and break the paired-initialisation check below.
x = torch.randn(2, 3, 32, 32)
model.eval()
with torch.no_grad():
  out = model.cpu()(x)
assert out.shape == (2, 2)
print(f"forward ok: input {tuple(x.shape)} -> logits {tuple(out.shape)}")

#3. Paired initialisation within this architecture: same seed, same weights,
#checked over every state-dict tensor (parameters and buffers).
set_seed(42)
model_b = build_resnet18_cifar(pretrained=False)
sd_a, sd_b = model.state_dict(), model_b.state_dict()
assert sd_a.keys() == sd_b.keys()
for name in sd_a:
  assert torch.equal(sd_a[name], sd_b[name]), f"tensor differs: {name}"
print(f"paired init ok: all {len(sd_a)} state-dict tensors identical across two seed-42 builds")

#4. Corruption operators accept a 32x32 batch at each worst severity.
imgs = torch.rand(4, 3, 32, 32)
for kind, sev in [("noise", 0.10), ("blur", 1.0), ("jpeg", 40)]:
  c = corrupt(imgs, kind, sev)
  assert c.shape == imgs.shape and c.min() >= 0.0 and c.max() <= 1.0
print("corruption ops ok: noise/blur/jpeg at worst severity preserve shape and range")

#5. Artefact audit, read-only. A completed matrix legitimately leaves every
#planned path present, so blanket disjointness is the wrong invariant. The
#right one: any planned ResNet path that exists must belong to a complete
#bundle that passes the same validation gate the resume path uses; partial,
#invalid, config-drifted or wrong-provenance bundles fail closed. Nothing
#here writes, deletes or renames - completed Drive artefacts are untouchable.
LEGITIMATE_VERSIONS = {"3", "unversioned"}  #kagglehub release 3, or the Colab Kaggle mount


def recorded_dataset_meta(hist_path):
  """The dataset identity a bundle recorded for itself, accepted only from a
  legitimate provenance. The bundle is then gated against its own recorded
  identity, so cross-provenance mixing still surfaces as dataset drift while
  both legitimate provenances (local kagglehub, Colab mount) validate."""
  with open(hist_path) as f:
    saved = json.load(f).get("dataset", {})
  version = saved.get("cifake_version")
  assert version in LEGITIMATE_VERSIONS, (
    f"{os.path.basename(hist_path)}: unrecognised cifake_version {version!r}")
  assert saved.get("val_frac") == VAL_FRAC, (
    f"{os.path.basename(hist_path)}: val_frac drift {saved.get('val_frac')!r}")
  return {"cifake_version": version, "val_frac": VAL_FRAC}


def audit_existing_artefacts():
  """Audit RESULTS_DIR without modifying it. Per strategy x seed pair: absent
  is fine, complete and gate-valid is fine, anything else raises. The
  aggregate may exist only above six valid bundles and must be exactly the
  96-row matrix. Crashed-session .tmp leftovers fail the audit rather than
  being cleaned up. Returns (n_valid_bundles, aggregate_rows)."""
  n_valid = 0
  for strategy, overrides in STRATEGIES.items():
    config = {**EXPERIMENT_CONFIG, **overrides}
    for seed in SEEDS:
      finals = run_paths(strategy, seed)
      present = [p for p in finals if os.path.exists(p)]
      if not present:
        continue
      assert len(present) == 3, (
        f"partial bundle {strategy} seed {seed}: only "
        f"{[os.path.basename(p) for p in present]} present")
      meta = recorded_dataset_meta(finals[1])
      df, reason = validate_bundle(*finals, strategy, seed, config, meta)
      assert df is not None, f"invalid bundle {strategy} seed {seed}: {reason}"
      n_valid += 1

  leftovers = [f for f in os.listdir(RESULTS_DIR)
         if f.startswith(ARTEFACT_PREFIX) and f.endswith(".tmp")] \
    if os.path.isdir(RESULTS_DIR) else []
  assert not leftovers, f"crashed-session temporaries present: {leftovers}"

  agg_path = os.path.join(RESULTS_DIR, f"{ARTEFACT_PREFIX}all_results.csv")
  agg_rows = 0
  if os.path.exists(agg_path):
    n_expected = len(STRATEGIES) * len(SEEDS)
    assert n_valid == n_expected, (
      f"aggregate exists but only {n_valid}/{n_expected} bundles are valid")
    agg = pd.read_csv(agg_path)
    assert agg.shape[0] == n_expected * 16, f"aggregate has {agg.shape[0]} rows, expected {n_expected * 16}"
    assert (agg["model"] == MODEL_NAME).all(), "aggregate contains foreign model rows"
    assert set(zip(agg["strategy"], agg["seed"])) == \
      {(s, sd) for s in STRATEGIES for sd in SEEDS}, "aggregate strategy x seed set wrong"
    assert not agg[METRIC_COLS].isna().any().any()
    agg_rows = agg.shape[0]
  return n_valid, agg_rows


n_valid, agg_rows = audit_existing_artefacts()

#Path guard is unchanged: every planned path carries the prefix, and a CNN
#filename is still rejected outright.
resnet_paths = {p for s in STRATEGIES for seed in SEEDS for p in run_paths(s, seed)}
resnet_paths.add(os.path.join(RESULTS_DIR, f"{ARTEFACT_PREFIX}all_results.csv"))
for p in resnet_paths:
  assert_safe_path(p)
try:
  assert_safe_path(os.path.join(RESULTS_DIR, "baseline_seed42.pt"))
  guard_raised = False
except AssertionError:
  guard_raised = True
assert guard_raised, "guard failed to reject a CNN artefact path"
print(f"artefact audit ok: {n_valid} valid existing bundles, "
    f"aggregate {f'{agg_rows} rows' if agg_rows else 'absent'}; "
    "partial/invalid bundles fail closed; CNN path rejected")

#6. Publication pipeline, exercised in a temp directory so the real results/
#is never touched: a fabricated valid bundle publishes atomically and is then
#accepted by the resume gate; a simulated crash between renames leaves a
#partial bundle that fails closed; write-once still rejects a second write.
import tempfile
_real_results_dir = RESULTS_DIR
_test_config = {**EXPERIMENT_CONFIG, "augment": False}
_test_meta = {"cifake_version": "3", "val_frac": VAL_FRAC}


def _fabricated_bundle(strategy, seed):
  """An untrained but gate-valid bundle: real state_dict, protocol-shaped
  history and a 16-condition csv with in-range metrics."""
  history = {"epochs_run": 1, "best_epoch": 1, "best_val_loss": 0.5,
       "train_losses": [0.5], "val_losses": [0.5],
       "config": dict(_test_config), "dataset": dict(_test_meta)}
  rows = [{"model": MODEL_NAME, "strategy": strategy, "corruption": kind,
       "severity": sev, "seed": seed, "accuracy": 0.5, "precision": 0.5,
       "recall": 0.5, "f1": 0.5, "roc_auc": 0.5, "epochs_run": 1,
       "best_epoch": 1, "lr": _test_config["lr"], "augment": _test_config["augment"]}
      for kind, sev in CONDITIONS]
  return model.state_dict(), history, pd.DataFrame(rows)


try:
  with tempfile.TemporaryDirectory() as tmp:
    RESULTS_DIR = tmp

    #6a. Happy path: publish validates in temp files then renames; the gate
    #accepts the result and no .tmp files remain.
    state, hist, fake_df = _fabricated_bundle("baseline", 42)
    publish_bundle("baseline", 42, state, hist, fake_df, _test_config, _test_meta)
    accepted, reason = check_saved_run("baseline", 42, _test_config, _test_meta)
    assert accepted is not None, f"published bundle rejected: {reason}"
    assert not [f for f in os.listdir(tmp) if f.endswith(".tmp")], "temp files left behind"

    #6b. Simulated crash between renames: temps written and validated, but
    #only the checkpoint reached its final name. The resume must fail closed
    #and the preflight must refuse to run this pair again.
    finals = run_paths("robust", 42)
    state, hist, fake_df = _fabricated_bundle("robust", 42)
    torch.save(state, finals[0])  #crash after first os.replace: ckpt final, rest lost
    try:
      resolve_existing_bundle("robust", 42, {**EXPERIMENT_CONFIG, "augment": True}, _test_meta)
      failed_closed = False
    except RuntimeError:
      failed_closed = True
    assert failed_closed, "partial publication did not fail closed"
    try:
      preflight_publication("robust", 42)
      preflight_held = False
    except AssertionError:
      preflight_held = True
    assert preflight_held, "preflight allowed rerun over a partial bundle"

    #6c. Write-once: the successfully published bundle refuses a second write.
    try:
      assert_write_once(run_paths("baseline", 42)[0])
      write_once_held = False
    except AssertionError:
      write_once_held = True
    assert write_once_held, "existing resnet18_ artefact was not protected"

    #6d. An invalid bundle must never reach final names: publishing with a
    #wrong-config history raises, quarantines all three temporaries with the
    #reason, and leaves nothing at final names or as .tmp.
    state, hist, fake_df = _fabricated_bundle("baseline", 43)
    hist["config"] = {**_test_config, "lr": 0.01}  #config drift
    try:
      publish_bundle("baseline", 43, state, hist, fake_df, _test_config, _test_meta)
      validation_held = False
    except RuntimeError as err:
      validation_held = True
      assert "quarantine" in str(err) or "preserved" in str(err)
    assert validation_held, "invalid bundle was published"
    assert not any(os.path.exists(p) for p in run_paths("baseline", 43)), \
      "invalid publication left files at final names"
    assert not [f for f in os.listdir(tmp) if f.endswith(".tmp")], "temp files left behind"

    qroot = os.path.join(tmp, "resnet18_quarantine")
    qdirs = sorted(os.listdir(qroot))
    assert len(qdirs) == 1 and qdirs[0].endswith("_baseline_seed43")
    qfiles = set(os.listdir(os.path.join(qroot, qdirs[0])))
    assert qfiles == {"resnet18_baseline_seed43.pt.tmp",
              "resnet18_baseline_seed43_history.json.tmp",
              "resnet18_baseline_seed43_results.csv.tmp",
              "REASON.txt"}, qfiles
    with open(os.path.join(qroot, qdirs[0], "REASON.txt")) as f:
      assert "config drift" in f.read()

    #6e. Quarantined files are never resumable or valid: the gate still sees
    #this pair as absent, and a rerun would be allowed only because the final
    #and .tmp paths are clear, never by reading quarantine.
    accepted, reason = check_saved_run("baseline", 43, _test_config, _test_meta)
    assert accepted is None and reason.startswith("missing")
    preflight_publication("baseline", 43)  #must not raise

    #6f. Successful publication quarantines nothing: only the one directory
    #from 6d exists, so the happy paths of 6a left no quarantine behind.
    assert sorted(os.listdir(qroot)) == qdirs, "success path created quarantine entries"

    #6g. Stale aggregate temp from a crashed session is discarded and the
    #aggregate is replaced atomically with the full rebuild.
    stale = os.path.join(tmp, f"{ARTEFACT_PREFIX}all_results.csv.tmp")
    with open(stale, "w") as f:
      f.write("stale junk from a dead session")
    combined = write_aggregate(fake_df)
    assert not os.path.exists(stale), "stale aggregate temp survived"
    assert len(pd.read_csv(combined)) == 16
finally:
  RESULTS_DIR = _real_results_dir
print("publication ok: valid bundle publishes atomically and passes the gate; "
    "crash between renames fails closed and blocks rerun; second write refused; "
    "invalid bundle quarantined with reason, never resumable; success leaves no "
    "quarantine; stale aggregate temp discarded and aggregate replaced atomically")

#7. Dataset root classification, dataset-free: fabricated directory skeletons
#for both legitimate path forms, a malformed layout, and an unrecognised root.
def _make_layout(root):
  for split, cls in CIFAKE_CLASS_DIRS:
    os.makedirs(os.path.join(root, split, cls))

with tempfile.TemporaryDirectory() as tmp:
  #7a. Versioned kagglehub cache form: versions/3 with the full class layout.
  kh_root = os.path.join(tmp, "kagglehub", "versions", "3")
  _make_layout(kh_root)
  meta = dataset_meta_for_root(kh_root)
  assert meta == {"cifake_version": "3", "provenance": "kagglehub_versioned",
          "version_available": True, "path": kh_root}, meta

  #7b. Unversioned Colab mount form: directory named by the dataset slug.
  mount_root = os.path.join(tmp, CIFAKE_SLUG)
  _make_layout(mount_root)
  meta = dataset_meta_for_root(mount_root)
  assert meta == {"cifake_version": "unversioned",
          "provenance": "kaggle_colab_mount",
          "version_available": False, "path": mount_root}, meta

  #7c. Wrong kagglehub release must be refused.
  bad_version = os.path.join(tmp, "kagglehub2", "versions", "4")
  _make_layout(bad_version)
  try:
    dataset_meta_for_root(bad_version)
    version_held = False
  except AssertionError:
    version_held = True
  assert version_held, "release 4 was accepted as release 3"

  #7d. Malformed structure: a missing class directory fails with its name.
  broken = os.path.join(tmp, "kagglehub3", "versions", "3")
  _make_layout(broken)
  os.rmdir(os.path.join(broken, "test", "FAKE"))
  try:
    dataset_meta_for_root(broken)
    layout_held = False
  except RuntimeError as err:
    layout_held = "test/FAKE" in str(err)
  assert layout_held, "malformed layout not reported"

  #7e. Neither form: an arbitrary directory is rejected outright.
  stray = os.path.join(tmp, "somewhere", "else")
  _make_layout(stray)
  try:
    dataset_meta_for_root(stray)
    stray_held = False
  except RuntimeError:
    stray_held = True
  assert stray_held, "unrecognised root was accepted"

#7f. Provenance is gated: a bundle recorded against the unversioned mount is
#rejected by a session validating against release 3, and vice versa. The two
#forms never silently mix.
_real_results_dir = RESULTS_DIR
try:
  with tempfile.TemporaryDirectory() as tmp:
    RESULTS_DIR = tmp
    mount_meta = {"cifake_version": "unversioned", "val_frac": VAL_FRAC}
    state, hist, fake_df = _fabricated_bundle("baseline", 42)
    hist["dataset"] = dict(mount_meta)
    publish_bundle("baseline", 42, state, hist, fake_df, _test_config, mount_meta)
    accepted, reason = check_saved_run("baseline", 42, _test_config, _test_meta)
    assert accepted is None and "dataset drift" in reason, reason
    accepted, reason = check_saved_run("baseline", 42, _test_config, mount_meta)
    assert accepted is not None, f"same-provenance bundle rejected: {reason}"
finally:
  RESULTS_DIR = _real_results_dir
print("dataset roots ok: versioned kagglehub and unversioned Colab mount both "
    "classified with provenance; wrong release, malformed layout and stray "
    "roots refused; cross-provenance bundles rejected by the gate")

#8. The artefact audit itself, against fabricated bundles in a temp directory
#so the real results are never touched: valid states pass, every corrupt
#state fails closed, and the audit never modifies a file.
_real_results_dir = RESULTS_DIR
_robust_config = {**EXPERIMENT_CONFIG, "augment": True}


def _audit_raises(fragment):
  try:
    audit_existing_artefacts()
    return False
  except AssertionError as err:
    return fragment in str(err)


try:
  with tempfile.TemporaryDirectory() as tmp:
    RESULTS_DIR = tmp

    #8a. Empty directory: nothing to validate, nothing to refuse.
    assert audit_existing_artefacts() == (0, 0)

    #8b. One published valid bundle is accepted.
    state, hist, fake_df = _fabricated_bundle("baseline", 42)
    publish_bundle("baseline", 42, state, hist, fake_df, _test_config, _test_meta)
    assert audit_existing_artefacts() == (1, 0)

    #8c. A partial bundle (checkpoint only at final names) fails closed.
    state, hist, fake_df = _fabricated_bundle("robust", 42)
    torch.save(state, run_paths("robust", 42)[0])
    assert _audit_raises("partial bundle robust seed 42"), "partial bundle accepted"
    os.remove(run_paths("robust", 42)[0])  #test scaffolding only, never Drive artefacts

    #8d. Config drift at final names fails closed: files written directly,
    #because publish_bundle would rightly refuse to produce this state.
    state, hist, fake_df = _fabricated_bundle("robust", 42)
    hist["config"] = {**_robust_config, "lr": 0.01}
    ck, hp, cp = run_paths("robust", 42)
    torch.save(state, ck)
    with open(hp, "w") as f:
      json.dump(hist, f)
    fake_df.to_csv(cp, index=False)
    assert _audit_raises("invalid bundle robust seed 42"), "config drift accepted"

    #8e. Wrong provenance fails closed before the gate even runs.
    hist["config"] = dict(_robust_config)
    hist["dataset"] = {"cifake_version": "4", "val_frac": VAL_FRAC}
    with open(hp, "w") as f:
      json.dump(hist, f)
    assert _audit_raises("unrecognised cifake_version"), "wrong provenance accepted"

    #8f. Colab-mount provenance is legitimate and validates.
    hist["dataset"] = {"cifake_version": "unversioned", "val_frac": VAL_FRAC}
    with open(hp, "w") as f:
      json.dump(hist, f)
    assert audit_existing_artefacts() == (2, 0)

    #8g. Aggregate above fewer than six valid bundles fails closed.
    agg_path = os.path.join(tmp, f"{ARTEFACT_PREFIX}all_results.csv")
    pd.concat([fake_df] * 3).to_csv(agg_path, index=False)
    assert _audit_raises("aggregate exists but only 2/6"), "premature aggregate accepted"
    os.remove(agg_path)

    #8h. The full happy state: six valid bundles plus the 96-row aggregate.
    frames = [pd.read_csv(run_paths("baseline", 42)[2]), pd.read_csv(cp)]
    for strategy, seed in [("baseline", 43), ("baseline", 44),
                ("robust", 43), ("robust", 44)]:
      state, hist, fake_df = _fabricated_bundle(strategy, seed)
      if strategy == "robust":
        hist["config"] = {**EXPERIMENT_CONFIG, "augment": True}
        fake_df["augment"] = True
      cfg = {**EXPERIMENT_CONFIG, **STRATEGIES[strategy]}
      publish_bundle(strategy, seed, state, hist, fake_df, cfg, _test_meta)
      frames.append(fake_df)
    pd.concat(frames, ignore_index=True).to_csv(agg_path, index=False)
    assert audit_existing_artefacts() == (6, 96)

    #8i. A crashed-session temporary fails the audit rather than being cleaned.
    stray = os.path.join(tmp, f"{ARTEFACT_PREFIX}robust_seed43.pt.tmp")
    with open(stray, "w") as f:
      f.write("crash leftover")
    assert _audit_raises("crashed-session temporaries"), "stray .tmp accepted"
finally:
  RESULTS_DIR = _real_results_dir
print("artefact audit checks ok: empty, single-valid and full six-plus-aggregate "
    "states accepted; partial bundle, config drift, wrong provenance, premature "
    "aggregate and crashed-session temporaries all fail closed")

print("\nall structural checks passed")

stem ok: conv1 3x3 stride 1 no bias, no max-pool, 2-class head, 11,169,858 parameters
forward ok: input (2, 3, 32, 32) -> logits (2, 2)


paired init ok: all 122 state-dict tensors identical across two seed-42 builds
corruption ops ok: noise/blur/jpeg at worst severity preserve shape and range
artefact audit ok: 0 valid existing bundles, aggregate absent; partial/invalid bundles fail closed; CNN path rejected


discarding stale aggregate temp resnet18_all_results.csv.tmp
publication ok: valid bundle publishes atomically and passes the gate; crash between renames fails closed and blocks rerun; second write refused; invalid bundle quarantined with reason, never resumable; success leaves no quarantine; stale aggregate temp discarded and aggregate replaced atomically


dataset roots ok: versioned kagglehub and unversioned Colab mount both classified with provenance; wrong release, malformed layout and stray roots refused; cross-provenance bundles rejected by the gate


artefact audit checks ok: empty, single-valid and full six-plus-aggregate states accepted; partial bundle, config drift, wrong provenance, premature aggregate and crashed-session temporaries all fail closed

all structural checks passed


In [10]:
#Completeness summary for this session's staged scope. Read from disk through
#the same validation gate the resume path uses, so "complete" here means a
#bundle the next session would accept, not merely files that exist.
print(f"staged scope: seeds {STAGED_SEEDS}, strategies {list(STRATEGIES)}")

n_expected = len(STAGED_SEEDS) * len(STRATEGIES)
n_complete = 0

for seed in STAGED_SEEDS:
  for strategy, overrides in STRATEGIES.items():
    config = {**EXPERIMENT_CONFIG, **overrides}
    #Gate each bundle against its own recorded provenance (release 3 locally,
    #unversioned on the Colab mount), the same rule the artefact audit uses.
    hist_p = run_paths(strategy, seed)[1]
    try:
      _dataset_meta = recorded_dataset_meta(hist_p) if os.path.exists(hist_p) \
        else {"cifake_version": "3", "val_frac": VAL_FRAC}
    except AssertionError as err:
      print(f"  {strategy} seed {seed}: INCOMPLETE (bad provenance: {err})")
      continue
    df, reason = check_saved_run(strategy, seed, config, _dataset_meta)
    if df is None:
      print(f"  {strategy} seed {seed}: INCOMPLETE ({reason})")
      continue
    n_complete += 1
    hist_path = run_paths(strategy, seed)[1]
    with open(hist_path) as f:
      history = json.load(f)
    env = history.get("environment", {})
    clean_acc = float(df.loc[df["corruption"] == "none", "accuracy"].iloc[0])
    print(f"  {strategy} seed {seed}: complete, {len(df)} rows, "
        f"epochs {history['epochs_run']} (best {history['best_epoch']}), "
        f"clean acc {clean_acc:.4f}, wall {history.get('wall_time_s', '?')}s")
    print(f"    gpu {env.get('gpu')}, cuda {env.get('cuda')}, "
        f"torch {env.get('torch')}, params {env.get('param_count'):,}"
        if env.get("param_count") else f"    environment: {env or 'not recorded'}")

agg = os.path.join(RESULTS_DIR, f"{ARTEFACT_PREFIX}all_results.csv")
agg_note = f"{len(pd.read_csv(agg))} rows" if os.path.exists(agg) else "not written (needs all 6 bundles)"
print(f"summary: {n_complete}/{n_expected} staged bundles complete and gate-valid; aggregate: {agg_note}")

staged scope: seeds [42, 43, 44], strategies ['baseline', 'robust']
  baseline seed 42: INCOMPLETE (missing resnet18_baseline_seed42.pt, resnet18_baseline_seed42_history.json, resnet18_baseline_seed42_results.csv)
  robust seed 42: INCOMPLETE (missing resnet18_robust_seed42.pt, resnet18_robust_seed42_history.json, resnet18_robust_seed42_results.csv)
  baseline seed 43: INCOMPLETE (missing resnet18_baseline_seed43.pt, resnet18_baseline_seed43_history.json, resnet18_baseline_seed43_results.csv)
  robust seed 43: INCOMPLETE (missing resnet18_robust_seed43.pt, resnet18_robust_seed43_history.json, resnet18_robust_seed43_results.csv)
  baseline seed 44: INCOMPLETE (missing resnet18_baseline_seed44.pt, resnet18_baseline_seed44_history.json, resnet18_baseline_seed44_results.csv)
  robust seed 44: INCOMPLETE (missing resnet18_robust_seed44.pt, resnet18_robust_seed44_history.json, resnet18_robust_seed44_results.csv)
summary: 0/6 staged bundles complete and gate-valid; aggregate: not written (nee